In [1]:

import os
from pathlib import Path

SEARCH_ROOT = Path("/kaggle/input")

wanted = {
    "unet_resnet34.pt",
    "unet_resnet34_last.pt",
    "detector_best.pt",
    "detector_last.pt",
    "best.pt",
    "last.pt",
    "unet_training.log",
    "detector_training.log",
    "training_manifest.json",
}

found = []

for root, dirs, files in os.walk(SEARCH_ROOT):
    # Avoid unnecessary traversal into hidden folders
    dirs[:] = [d for d in dirs if not d.startswith(".")]

    for filename in files:
        if filename in wanted:
            path = Path(root) / filename
            found.append(path)
            print(f"FOUND: {path}  ({path.stat().st_size / 1024**2:.2f} MB)")

print("\nTotal matching files:", len(found))

if not found:
    print("\nNo saved checkpoints found in /kaggle/input.")
    print("You may need to attach your previous notebook output as a Kaggle Dataset.")


FOUND: /kaggle/input/datasets/hassanraza16/nnnnnnnnnnn/filament_outputs/detector_training.log  (1.36 MB)
FOUND: /kaggle/input/datasets/hassanraza16/nnnnnnnnnnn/filament_outputs/training_manifest.json  (0.00 MB)
FOUND: /kaggle/input/datasets/hassanraza16/nnnnnnnnnnn/filament_outputs/unet_resnet34.pt  (93.37 MB)
FOUND: /kaggle/input/datasets/hassanraza16/nnnnnnnnnnn/filament_outputs/unet_resnet34_last.pt  (93.37 MB)
FOUND: /kaggle/input/datasets/hassanraza16/nnnnnnnnnnn/filament_outputs/unet_training.log  (0.02 MB)
FOUND: /kaggle/input/datasets/hassanraza16/nnnnnnnnnnn/filament_outputs/detector/detector_best.pt  (19.64 MB)
FOUND: /kaggle/input/datasets/hassanraza16/nnnnnnnnnnn/filament_outputs/detector/detector_last.pt  (19.64 MB)
FOUND: /kaggle/input/datasets/hassanraza16/nnnnnnnnnnn/filament_outputs/detector/train/weights/best.pt  (19.64 MB)
FOUND: /kaggle/input/datasets/hassanraza16/nnnnnnnnnnn/filament_outputs/detector/train/weights/last.pt  (19.64 MB)

Total matching files: 9


In [2]:

import shutil
from pathlib import Path

DEST = Path("/kaggle/working/filament_outputs_recovered")
DEST.mkdir(parents=True, exist_ok=True)

# Prefer the specifically named detector checkpoints.
# Avoid accidentally replacing them with generic best.pt/last.pt.
preferred = {
    "unet_resnet34.pt",
    "unet_resnet34_last.pt",
    "detector_best.pt",
    "detector_last.pt",
    "unet_training.log",
    "detector_training.log",
    "training_manifest.json",
}

for src in found:
    if src.name in preferred:
        target = DEST / src.name
        shutil.copy2(src, target)
        print("COPIED:", target)

print("\nRecovered files:")
for p in sorted(DEST.iterdir()):
    print(p.name, f"{p.stat().st_size / 1024**2:.2f} MB")


COPIED: /kaggle/working/filament_outputs_recovered/detector_training.log
COPIED: /kaggle/working/filament_outputs_recovered/training_manifest.json
COPIED: /kaggle/working/filament_outputs_recovered/unet_resnet34.pt
COPIED: /kaggle/working/filament_outputs_recovered/unet_resnet34_last.pt
COPIED: /kaggle/working/filament_outputs_recovered/unet_training.log
COPIED: /kaggle/working/filament_outputs_recovered/detector_best.pt
COPIED: /kaggle/working/filament_outputs_recovered/detector_last.pt

Recovered files:
detector_best.pt 19.64 MB
detector_last.pt 19.64 MB
detector_training.log 1.36 MB
training_manifest.json 0.00 MB
unet_resnet34.pt 93.37 MB
unet_resnet34_last.pt 93.37 MB
unet_training.log 0.02 MB


In [3]:

from pathlib import Path

out = Path("/kaggle/working/filament_outputs_recovered")

required = [
    "unet_resnet34.pt",
    "detector_best.pt",
]

for name in required:
    p = out / name
    print(f"{name}: {'READY' if p.exists() else 'MISSING'}")

assert all((out / name).exists() for name in required), \
    "A required checkpoint is missing. Do not start inference yet."

print("\nBoth main checkpoints are ready for inference.")


unet_resnet34.pt: READY
detector_best.pt: READY

Both main checkpoints are ready for inference.


In [4]:

import os, sys, subprocess, shutil, json, time
from pathlib import Path

DATA = Path("/kaggle/input/competitions/filament-segmentation-2026/MAGFiLO_1.0_Kaggle_2026")
assert DATA.exists(), f"Competition dataset missing: {DATA}"

RECOVERED = Path("/kaggle/working/filament_outputs_recovered")
UNET = RECOVERED / "unet_resnet34.pt"
DETECTOR = RECOVERED / "detector_best.pt"

assert UNET.exists(), f"Missing checkpoint: {UNET}"
assert DETECTOR.exists(), f"Missing checkpoint: {DETECTOR}"

REPO = Path("/kaggle/working/filament-seg")
if REPO.exists():
    shutil.rmtree(REPO)

subprocess.run([
    "git", "clone", "--depth", "1",
    "https://github.com/ameyypawar/filament-seg.git",
    str(REPO)
], check=True)

assert (REPO / "scripts" / "cache_logits.py").exists()
assert (REPO / "scripts" / "yolo_predict.py").exists()
assert (REPO / "scripts" / "fuse_detections.py").exists()

WORK = Path("/kaggle/working/filament_inference")
OUTPUT = WORK / "outputs"
CACHE = REPO / "data" / "cache"
LOGITS = WORK / "logits"
DETECTIONS = WORK / "detections"

for p in [WORK, OUTPUT, CACHE, LOGITS, DETECTIONS]:
    p.mkdir(parents=True, exist_ok=True)

os.environ["FILAMENT_DATA_ROOT"] = str(DATA)
os.environ["FILAMENT_OUTPUT_ROOT"] = str(OUTPUT)
os.environ["PYTHONUNBUFFERED"] = "1"

subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "segmentation-models-pytorch==0.5.0",
    "ultralytics==8.4.173",
    "pycocotools==2.0.10"
], check=True)

import torch
print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("U-Net:", UNET)
print("Detector:", DETECTOR)
print("Setup successful.")


Cloning into '/kaggle/working/filament-seg'...


GPU available: True
GPU: Tesla T4
U-Net: /kaggle/working/filament_outputs_recovered/unet_resnet34.pt
Detector: /kaggle/working/filament_outputs_recovered/detector_best.pt
Setup successful.


In [5]:

def run_cmd(args):
    args = [str(x) for x in args]
    print("\nRUNNING:", " ".join(args), flush=True)
    subprocess.run(args, cwd=REPO, check=True)

run_cmd([
    sys.executable, "scripts/preprocess.py",
    "--workers", "4"
])

run_cmd([
    sys.executable, "scripts/make_splits.py",
    "--group-by", "date",
    "--out", OUTPUT / "splits.json"
])

assert (CACHE / "disk.json").exists()
assert (CACHE / "flat").exists()
assert (OUTPUT / "splits.json").exists()

print("Preprocessing and split completed.")



RUNNING: /usr/bin/python3 scripts/preprocess.py --workers 4
887 observations (707 train, 180 test)
  25/887
  50/887
  75/887
  100/887
  125/887
  150/887
  175/887
  200/887
  225/887
  250/887
  275/887
  300/887
  325/887
  350/887
  375/887
  400/887
  425/887
  450/887
  475/887
  500/887
  525/887
  550/887
  575/887
  600/887
  625/887
  650/887
  675/887
  700/887
  725/887
  750/887
  775/887
  800/887
  825/887
  850/887
  875/887
  887/887
wrote 887 disk geometries -> /kaggle/working/filament-seg/data/cache/disk.json
wrote 1154 new mask PNGs -> /kaggle/working/filament-seg/data/cache/mask
wrote 707 new consensus PNGs -> /kaggle/working/filament-seg/data/cache/consensus

RUNNING: /usr/bin/python3 scripts/make_splits.py --group-by date --out /kaggle/working/filament_inference/outputs/splits.json
group_by=date  groups=656
train: 906 views / 563 observations
val  : 248 views / 144 observations
observation overlap between train and val: 0 (must be 0)
wrote /kaggle/working/filam

In [6]:

DET_VAL = DETECTIONS / "detections_val.json"
DET_TEST = DETECTIONS / "detections_test.json"

for subset, destination in [
    ("val", DET_VAL),
    ("test", DET_TEST)
]:
    run_cmd([
        sys.executable, "scripts/yolo_predict.py",
        "--weights", DETECTOR,
        "--subset", subset,
        "--out", destination,
        "--imgsz", "1024",
        "--conf", "0.05",
        "--iou", "0.6",
        "--max-det", "150",
        "--device", "0",
        "--tta", "none"
    ])

assert DET_VAL.exists() and DET_TEST.exists()
print("YOLO predictions completed.")



RUNNING: /usr/bin/python3 scripts/yolo_predict.py --weights /kaggle/working/filament_outputs_recovered/detector_best.pt --subset val --out /kaggle/working/filament_inference/detections/detections_val.json --imgsz 1024 --conf 0.05 --iou 0.6 --max-det 150 --device 0 --tta none
Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
  25/144
  50/144
  75/144
  100/144
  125/144
  144/144
wrote /kaggle/working/filament_inference/detections/detections_val.json: 2396 detections over 144 observations

RUNNING: /usr/bin/python3 scripts/yolo_predict.py --weights /kaggle/working/filament_outputs_recovered/detector_best.pt --subset test --out /kaggle/working/filament_inference/detections/detections_test.json --imgsz 1024 --conf 0.05 --iou 0.6 --max-det 150 --dev

In [7]:

LOGIT_DIR = LOGITS / "unet"
SUBMISSION = Path("/kaggle/working/submission_hybrid.csv")
REPORT = WORK / "fusion_validation.json"

run_cmd([
    sys.executable, "scripts/fuse_detections.py",
    "--checkpoint", UNET,
    "--logit-dir", LOGIT_DIR,
    "--detections-val", DET_VAL,
    "--detections-test", DET_TEST,
    "--baseline",
    "threshold=0.6,min_area=400,bridge_gap=24,close_radius=3,open_radius=0",

    "--tta", "none",
    "--n-images", "72",
    "--threshold", "0.45", "0.50", "0.55",
    "--close-radius", "3",
    "--grow", "4", "8",
    "--min-score", "0.20", "0.30", "0.40",
    "--min-area", "150", "200",
    "--keep-unclaimed", "0",
    "--unclaimed-gap", "24",
    "--holdout-top", "3",
    "--workers", "2",
    "--out", REPORT,
    "--submission", SUBMISSION
])

assert SUBMISSION.exists(), "Fusion did not create the submission."

print("\nFusion validation report:")
print(REPORT.read_text()[:5000])
print("\nSubmission:", SUBMISSION)



RUNNING: /usr/bin/python3 scripts/fuse_detections.py --checkpoint /kaggle/working/filament_outputs_recovered/unet_resnet34.pt --logit-dir /kaggle/working/filament_inference/logits/unet --detections-val /kaggle/working/filament_inference/detections/detections_val.json --detections-test /kaggle/working/filament_inference/detections/detections_test.json --baseline threshold=0.6,min_area=400,bridge_gap=24,close_radius=3,open_radius=0 --tta none --n-images 72 --threshold 0.45 0.50 0.55 --close-radius 3 --grow 4 8 --min-score 0.20 0.30 0.40 --min-area 150 200 --keep-unclaimed 0 --unclaimed-gap 24 --holdout-top 3 --workers 2 --out /kaggle/working/filament_inference/fusion_validation.json --submission /kaggle/working/submission_hybrid.csv
device: cuda
models: 1; computing logits for 144/144 observations
  5/144
  10/144
  15/144
  20/144
  25/144
  30/144
  35/144
  40/144
  45/144
  50/144
  55/144
  60/144
  65/144
  70/144
  75/144
  80/144
  85/144
  90/144
  95/144
  100/144
  105/144
  

In [8]:

import pandas as pd

df = pd.read_csv(SUBMISSION)

print("Columns:", df.columns.tolist())
print("Total predicted instances:", len(df))
print(df.head())

assert df.columns.tolist() == ["filament_id", "segmentation_rle"]
assert len(df) > 0
assert df["filament_id"].notna().all()
assert df["segmentation_rle"].notna().all()
assert (df["segmentation_rle"].astype(str).str.len() > 0).all()

# Check that predictions cover all 180 test images.
test_dir = DATA / "test" / "test_images"
test_stems = {p.stem for p in test_dir.glob("*") if p.is_file()}

predicted_stems = set(df["filament_id"].astype(str).str.rsplit("_", n=1).str[0])
print("Test images:", len(test_stems))
print("Images represented by predictions:", len(predicted_stems))
print("Images without predictions:", len(test_stems - predicted_stems))

# Preserve a copy in the working directory.
backup = Path("/kaggle/working/submission_hybrid_backup.csv")
df.to_csv(backup, index=False)

print("\nSUBMISSION FILE READY")
print("Main:", SUBMISSION)
print("Backup:", backup)
print("Size:", round(SUBMISSION.stat().st_size / 1024**2, 2), "MB")


Columns: ['filament_id', 'segmentation_rle']
Total predicted instances: 1271
          filament_id                                   segmentation_rle
0  20110120105534Ch_1  o_Vi21oo11O1O1O2gPNMan15\QNNbn13]QNMcn14\QNMen...
1  20110120105534Ch_2  Ukln12mo11N2O1N1000001O00O0100O1M30O2O100O0101...
2  20110130110334Ch_1  c_nd22lo1:G2N2O6J1O11OnPN^Ogn1`0WQNAjn1?TQNCln...
3  20110214175414Mh_1  hY_g22lo19H;F1N1O1N2O000OM4N20O100O1O2N1010N10...
4  20110214175414Mh_2  \]d[23fo18N2N2N2H7O2O001O0oNZOgRNh0Wm1ZOhRNf0W...
Test images: 180
Images represented by predictions: 176
Images without predictions: 4

SUBMISSION FILE READY
Main: /kaggle/working/submission_hybrid.csv
Backup: /kaggle/working/submission_hybrid_backup.csv
Size: 0.32 MB
